← [Overview](00_overview.ipynb)

# Averaging

`averaging` is the **only time-based typical-periods method** in tsam. It splits
the ordered period list into $k$ **consecutive equal-size blocks** — purely by
position, without looking at the values at all.

This places it in the **time-based / typical-periods** cell of the Hoffmann (2020)
taxonomy.

> **What averaging does NOT do:** true calendar time-slices (e.g. grouping all
> winter-weekdays together, or all summer-weekend days) are **not** built into
> tsam. The `averaging` method only produces consecutive-period blocks.
> If you need season × weekday grouping, build the assignment vector externally.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

import tsam
from tsam import ClusterConfig

pio.renderers.default = "notebook_connected"

# --------------------------------------------------------------------------
# Load the shared tiny dataset produced by 01_preprocessing (../tiny.csv).
# --------------------------------------------------------------------------
tiny = pd.read_csv("../tiny.csv", index_col=0, parse_dates=True)

# Real dataset
raw = pd.read_csv("../testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-01-01":"2010-02-11"]
UNITS = {"GHI": "W/m²", "T": "°C", "Wind": "m/s", "Load": "MW"}
print("tiny:", tiny.shape, "  real:", data.shape)

tiny: (24, 2)   real: (1008, 4)


---

## Mechanism

Given $N$ periods and target $k$ clusters, the assignment is purely positional:

* Block $0$: periods $0 \ldots \lfloor N/k \rfloor - 1$
* Block $1$: periods $\lfloor N/k \rfloor \ldots 2\lfloor N/k \rfloor - 1$
* …
* Any remainder is appended to the last block.

The **cluster representative** is the mean of the block members — hence "averaging".

**TSAM configuration for averaging:**

In [2]:
from tsam import ClusterConfig
import tsam

# Averaging: purely positional consecutive-block grouping.
# Time-based / typical periods — no value similarity involved.
# representation defaults to 'mean' (block average) for averaging.
cfg_averaging = ClusterConfig(method="averaging", representation="mean")
print(cfg_averaging)

# Full aggregate call:
# result = tsam.aggregate(
#     df,
#     n_clusters=k,
#     period_duration="1D",
#     cluster=cfg_averaging,
# )

ClusterConfig(include_period_sums=False, method='averaging', representation='mean', scale_by_column_means=False, solver='highs', use_duration_curves=False)


In [3]:
k_avg = 3
n_periods = 6
block_size = n_periods // k_avg
remainder  = n_periods - block_size * k_avg

print(f"k={k_avg}, N={n_periods}, block_size={block_size}, remainder={remainder}")

avg_assignments = []
for c in range(k_avg):
    avg_assignments.extend([c] * block_size)
if remainder > 0:
    avg_assignments.extend([k_avg - 1] * remainder)

print("\nAveraging assignments (purely positional — values not consulted):")
for i, a in enumerate(avg_assignments):
    print(f"  day_{i} -> cluster {a}")

print("\nNote: days 0 and 1 both happen to be sunny days that land in the same cluster,")
print("but this is by position, not similarity. The algorithm has no knowledge of values.")

k=3, N=6, block_size=2, remainder=0

Averaging assignments (purely positional — values not consulted):
  day_0 -> cluster 0
  day_1 -> cluster 0
  day_2 -> cluster 1
  day_3 -> cluster 1
  day_4 -> cluster 2
  day_5 -> cluster 2

Note: days 0 and 1 both happen to be sunny days that land in the same cluster,
but this is by position, not similarity. The algorithm has no knowledge of values.


In [4]:
result_avg_tiny = tsam.aggregate(
    tiny,
    n_clusters=3,
    period_duration="1D",
    cluster=ClusterConfig(method="averaging"),
)
print("Averaging (tiny) — assignments:", result_avg_tiny.cluster_assignments)
print("Cluster counts:", result_avg_tiny.cluster_counts)

Averaging (tiny) — assignments: [0 0 1 1 2 2]
Cluster counts: {0: 2.0, 1: 2.0, 2: 2.0}


In [5]:
result_avg = tsam.aggregate(
    data,
    n_clusters=6,
    period_duration="1D",
    cluster=ClusterConfig(method="averaging"),
)
print("Averaging (real) — weighted RMSE:", round(result_avg.accuracy.weighted_rmse, 4))
print("(typically higher error than feature-based methods — it ignores similarity)")

Averaging (real) — weighted RMSE: 0.1356
(typically higher error than feature-based methods — it ignores similarity)


### Averaging on the calendar

Because averaging groups by position, the clusters are simply consecutive equal-size blocks of the calendar — the same block shape as contiguous clustering, but chosen without ever looking at the values.

In [6]:
result_avg.plot.clusters_over_time(
    columns=["Load"], units=UNITS, title="Averaging: consecutive positional blocks"
)

In [7]:
# Compare averaging vs k-means: one week of Load reconstruction
result_km = tsam.aggregate(
    data, n_clusters=6, period_duration="1D", cluster=ClusterConfig(method="kmeans")
)

week = slice("2010-01-11", "2010-01-17")
frames = []
for name, res in [("original", None), ("averaging", result_avg), ("kmeans", result_km)]:
    s = data.loc[week, "Load"] if res is None else res.reconstructed.loc[week, "Load"]
    frames.append(pd.DataFrame({"time": s.index, "Load": s.values, "method": name}))

px.line(
    pd.concat(frames),
    x="time", y="Load", color="method",
    title="Averaging vs k-means — one week of Load reconstruction",
)

C:\Users\j.belina\AppData\Local\miniforge3\envs\tsam_improve_reworked_notebooks\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


---

**Up next:**
* [Segmentation](05_segmentation.ipynb) — reducing the number of timesteps within each period